<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/NLP_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/text_simulation.csv",   # change path if needed
    encoding='latin-1',
    header=None
)

df.columns = ["polarity", "id", "date", "flag", "user", "text"]

In [ ]:
!pip uninstall -y transformers accelerate tokenizers huggingface_hub -q

In [ ]:
!rm -rf /usr/local/lib/python3.12/dist-packages/transformers*
!rm -rf /usr/local/lib/python3.12/dist-packages/accelerate*
!rm -rf /usr/local/lib/python3.12/dist-packages/tokenizers*

In [ ]:
!pip install transformers==4.38.2 accelerate==0.27.2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU: Tesla T4


In [ ]:
import re
import emoji

def clean_text(text):
    text = emoji.demojize(str(text))
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.lower()

df['text'] = df['text'].apply(clean_text)

In [ ]:
df['label'] = df['polarity'].map({0: 0, 2: 1, 4: 2})
df = df[['text', 'label']].dropna()

In [ ]:
df = df.sample(10000)   # keep small for GPU training

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
import torch

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

model = model.to("cuda")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from tqdm import tqdm
import torch

epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch} Loss:", total_loss / len(train_loader))

Epoch 0: 100%|██████████| 500/500 [01:33<00:00,  5.35it/s, loss=0.392]


Epoch 0 Loss: 0.5179753329753876


Epoch 1: 100%|██████████| 500/500 [01:31<00:00,  5.47it/s, loss=0.189]

Epoch 1 Loss: 0.3239813761264086


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

model.eval()
preds = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(batch["labels"].cpu().numpy())

accuracy = accuracy_score(true_labels, preds)
print("Accuracy:", accuracy)

Accuracy: 0.7675


In [ ]:
model.save_pretrained("/content/text_model")
tokenizer.save_pretrained("/content/text_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/text_model/tokenizer_config.json',
 '/content/text_model/tokenizer.json')

In [ ]:
import torch

def predict(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    predicted_class = torch.argmax(logits, dim=1).item()

    labels = {
        0: "NEGATIVE",
        1: "NEUTRAL",
        2: "POSITIVE"
    }

    return labels[predicted_class]

In [ ]:
print(predict("I love this project"))
print(predict("i love you"))

POSITIVE
POSITIVE


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("text_model", 'zip', "/content/text_model")
files.download("text_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>